# Lecture des exports de l'ancien Rocky (étape A1)

Ce notebook se trouve dans `verification/` de l'archive. Il charge chaque export Parquet et CSV, vérifie les nombres de lignes contre `exports/manifest.json`, puis montre quelques lectures types.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ARCHIVE = Path.cwd().parent
EXPORTS = ARCHIVE / "exports"
manifest = json.loads((EXPORTS / "manifest.json").read_text(encoding="utf-8"))
print("Source :", manifest["source"], "—", len(manifest["tables"]), "tables")

In [ ]:
parquet, csv, controle = {}, {}, []
for table, info in manifest["tables"].items():
    parquet[table] = pd.read_parquet(EXPORTS / "parquet" / f"{table}.parquet")
    csv[table] = pd.read_csv(EXPORTS / "csv" / f"{table}.csv")
    controle.append({
        "catégorie": info["categorie"], "table": table, "manifeste": info["lignes"],
        "parquet": len(parquet[table]), "csv": len(csv[table]),
        "colonnes": parquet[table].shape[1],
    })
controle = pd.DataFrame(controle)
controle["ok"] = (controle["manifeste"] == controle["parquet"]) & (controle["manifeste"] == controle["csv"])
assert controle["ok"].all(), controle[~controle["ok"]]
controle

## Offres × scores × rattachements

In [ ]:
offres = parquet["job_offers"]
scores = parquet["job_matches"]
liens = parquet["profile_jobs"]
vue = (liens.merge(offres, left_on="job_id", right_on="id", suffixes=("", "_offre"))
            .merge(scores, on=["job_id", "profile_id"], how="left", suffixes=("", "_score")))
print(len(vue), "rattachements,", vue["score"].notna().sum(), "avec score")
vue.sort_values("score", ascending=False)[["profile_id", "job_title", "company_name", "source_name", "score"]].head(10)

## Mails et décisions Gmail

In [ ]:
mails = parquet["email_messages"]
mails.groupby(["classification", "processing_state"], dropna=False).size().rename("mails").reset_index()

## Candidatures et événements

In [ ]:
parquet["applications"]["status"].value_counts(dropna=False)

In [ ]:
parquet["application_events"].groupby(["event_type", "source"], dropna=False).size().rename("événements")